In [7]:
!pip install -q transformers sentencepiece accelerate gradio


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-base"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model = model.to(device)

print("FLAN-T5 model loaded successfully!")

Using device: cpu


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 model loaded successfully!


In [9]:

def summarize_log(log_text):

    if not log_text or not log_text.strip():
        return "Please enter a valid error log."

    prompt = f"""
    Analyze this software build or error log.

    Explain the following:
    1. Short error summary
    2. Main error
    3. Possible root cause
    4. Affected component
    5. Recommended solution
    6. Prevention tips

    Use simple English.
    Do not invent information.
    If the cause is uncertain, mention that.

    Error log:
    {log_text}

    Analysis:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=220,
        num_beams=4,
        early_stopping=True
    )

    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return result


print("Error summarization function created!")


Error summarization function created!


In [11]:
sample_log = """
[INFO] Starting application build...
[INFO] Installing dependencies...
[INFO] Running Python application...
[ERROR] ModuleNotFoundError:
No module named 'pandas'
[ERROR] Build failed.
[INFO] Process exited with status code 1.
"""

print(sample_log)


[INFO] Starting application build...
[INFO] Installing dependencies...
[INFO] Running Python application...
[ERROR] ModuleNotFoundError:
No module named 'pandas'
[ERROR] Build failed.
[INFO] Process exited with status code 1.



In [12]:
result = summarize_log(sample_log)

print("========== AI ERROR ANALYSIS ==========")
print(result)

========== AI ERROR ANALYSIS ==========
[INFO] Starting application build... [INFO] Installing dependencies... [INFO] Running Python application... [ERROR] ModuleNotFoundError: No module named 'pandas' [ERROR] Build failed. [INFO] Process exited with status code 1.


In [14]:
test_logs = {
    "Missing Package": """
    ModuleNotFoundError: No module named 'numpy'
    Build failed.
    """,

    "Syntax Error": """
    File "main.py", line 10
    print("Hello"
    SyntaxError: unexpected EOF while parsing
    """,

    "File Not Found": """
    FileNotFoundError:
    No such file or directory: 'config.json'
    Application failed to start.
    """
}

for name, log in test_logs.items():
    print("\n==========", name, "==========")
    print(summarize_log(log))


========== Missing Package ==========
1. ModuleNotFoundError: No module named 'numpy' Build failed.

========== Syntax Error ==========
1. main.py 2. main.py 3. Possible root cause 4. Affected component 5. Recommended solution 6. Prevention tips

========== File Not Found ==========
1. FileNotFoundError: No such file or directory: 'config.json' 2. Application failed to start.


In [15]:
import gradio as gr

def analyze_error(log_text):
    return summarize_log(log_text)

demo = gr.Interface(
    fn=analyze_error,
    inputs=gr.Textbox(
        lines=12,
        label="Enter Build/Error Log",
        placeholder="Paste your error log here..."
    ),
    outputs=gr.Textbox(
        lines=15,
        label="AI Error Analysis"
    ),
    title="LLM-Based Build/Error-Log Summarization",
    description=(
        "Analyze software errors using a free open-source "
        "Large Language Model."
    ),
    submit_btn="Summarize Error Log"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3856134d90ed91ebde.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded:

    with open(
        filename,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as file:

        log_text = file.read()

    print("\nAnalyzing:", filename)
    print(summarize_log(log_text))